# NFL passing-yard predictions and betting opportunities with our Nori model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Synthefy/synthefy-nori/blob/main/examples/notebooks/nori-nfl-passing-yards.ipynb)

This notebook reconstructs the method in [our NFL passing-yard blog](https://synthefy.com/blog/nori-nfl-quarterback-passing-yards): pull public football data, build historical and in-game features, predict passing-yard distributions with Nori, and evaluate opportunities using recorded Kalshi quotes.

**Run this notebook from scratch.** Feature creation lives in the companion `nfl_passing_yards_features.py`; Colab downloads a pinned, checksum-verified copy automatically. It downloads public data and model weights, not precomputed research predictions. The default scores all 18 regular-season weeks of 2025 at Q1 and halftime. A repeat run may reuse fingerprint-verified predictions computed locally.

## Before you run it

Use Python 3.11 and run cells in order. A Colab GPU is recommended: **Runtime → Change runtime type → GPU**. CPU works, but full-season inference can take hours. Set WEEKS=(1,) for a partial smoke test, or keep all 18 weeks for the full run. Kalshi is optional and unavailable market data does not discard forecasts.

The transformations, model and selection rules match the original experiment. Public data can change after publication; a fresh 2020 input differs from the original snapshot. The results below are measured, not hard-coded to the blog's 24.1%. The strategy was refined on 2025, so this is exploratory reproduction rather than independent validation.

## 1. Install the measured package versions
Run this before importing the packages. If Jupyter already imported different
versions, restart its kernel after installation and run again.

In [ ]:
import sys
import subprocess
import importlib.metadata as metadata

required = {
    "synthefy-nori": "0.19.0",
    "nflreadpy": "0.1.5",
    "numpy": "2.4.6",
    "pandas": "3.0.5",
    "polars": "1.44.0",
    "torch": "2.13.0",
    "matplotlib": "3.11.1",
    "requests": "2.34.2",
    "huggingface-hub": "1.28.0",
    "scipy": "1.17.1",
    "scikit-learn": "1.9.0",
    "pyarrow": "25.0.1",
}
missing = []
for package, wanted in required.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != wanted:
        missing.append(f"{package}=={wanted}")
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print({name: metadata.version(name) for name in required})

## 2. Configure the run

The companion Python module contains the historical and in-game feature transformations. The notebook keeps the model, strategy, and results visible.


In [ ]:
from __future__ import annotations
from pathlib import Path
import hashlib
import json
import os
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import torch
import inspect as inference_inspect
from IPython.display import display

CACHE_DIR = Path(os.environ.get("NFL_CACHE_DIR", "nfl_blog_cache"))
OUTPUT_DIR = Path(os.environ.get("NFL_OUTPUT_DIR", "nfl_blog_outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEEKS = tuple(range(1, 19))
DEVICE = os.environ.get(
    "NFL_DEVICE",
    "cuda:0"
    if "COLAB_RELEASE_TAG" in os.environ and torch.cuda.is_available()
    else "cpu",
)
RUN_MARKET_BACKTEST = True
EXECUTION_COST = 0.05
REFRESH_SOURCE_DATA = False
RESUME_PREDICTIONS = True
print({"weeks": WEEKS, "device": DEVICE, "kalshi": RUN_MARKET_BACKTEST})

### Load feature helpers

Use the local companion file, or download the pinned version when running in Colab.


In [ ]:
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

HELPER_REVISION = "cf5d5e2b8f58e0baba45be50330321c43fd62024"
HELPER_SHA256 = "cf63ea3e4811fa25c35d42afbcc519cccb091491d06dcd4316ba6b63d1caa342"
helper_path = Path("nfl_passing_yards_features.py")
if not helper_path.exists():
    repo_helper = Path("examples/notebooks") / helper_path
    if repo_helper.exists():
        helper_path = repo_helper
    else:
        url = f"https://raw.githubusercontent.com/Synthefy/synthefy-nori/{HELPER_REVISION}/examples/notebooks/{helper_path.name}"
        with urlopen(url, timeout=60) as response:
            helper_bytes = response.read()
        if hashlib.sha256(helper_bytes).hexdigest() != HELPER_SHA256:
            raise ValueError("Feature helper checksum mismatch")
        helper_path.write_bytes(helper_bytes)
sys.path.insert(0, str(helper_path.resolve().parent))
from nfl_passing_yards_features import pipeline_blog_feature_columns, pipeline_build_blog_features


### Pinned model, weekly inference and verified resume

In [ ]:
from __future__ import annotations

import hashlib as inference_hashlib

import io as inference_io

import json as inference_json

from importlib.metadata import version as inference_version

from pathlib import Path as inference_Path

import numpy as inference_np

import polars as inference_pl

from huggingface_hub import hf_hub_download as inference_hf_hub_download

from synthefy_nori import NoriRegressor as inference_NoriRegressor

inference_REVISION = "157d6be39b5ba8809e4296d50abf3f41f3b72947"

inference_CHECKPOINT_SHA256 = (
    "a13b2bc31d8db24d17bae6d04844e0adf669e446087b0b7a34c7b05045d61323"
)

inference_TARGET = "remaining_passing_yards"

inference_IDENTITY_COLUMNS = [
    "game_id",
    "season",
    "week",
    "kickoff_utc",
    "live_anchor_utc",
    "live_decision_utc",
    "team",
    "opponent_team",
    "actual_qb_id",
    "actual_qb_name",
    "official_passing_yards",
    "live_qb_attempts",
    "live_qb_passing_yards",
    inference_TARGET,
]


def inference_runtime_versions():
    return {
        name: inference_version(name)
        for name in (
            "synthefy-nori",
            "numpy",
            "torch",
            "polars",
            "scipy",
            "scikit-learn",
        )
    }


def inference_frame_sha256(frame):
    """Hash logical ordered values and schema, not Arrow buffer padding."""
    payload = {
        "schema": [(name, str(dtype)) for name, dtype in frame.schema.items()],
        "rows": frame.to_dicts(),
    }
    encoded = inference_json.dumps(
        payload, default=str, separators=(",", ":"), allow_nan=True
    ).encode()
    return inference_hashlib.sha256(encoded).hexdigest()


def inference_prediction_provenance(train, query, columns, device):
    """Fingerprint all numerical inputs, their order, identities and runtime."""
    model_inputs = [
        *[
            inference_pl.col(c).cast(inference_pl.Float32).alias(f"model_input_{i}")
            for i, c in enumerate(columns)
        ],
        *[
            inference_pl.col(c).cast(inference_pl.Float64)
            if c == inference_TARGET
            else inference_pl.col(c)
            for c in inference_IDENTITY_COLUMNS
        ],
    ]
    return {
        "cache_version": 3,
        "train_sha256": inference_frame_sha256(train.select(model_inputs)),
        "query_sha256": inference_frame_sha256(query.select(model_inputs)),
        "selected_columns": list(columns),
        "checkpoint_sha256": inference_CHECKPOINT_SHA256,
        "checkpoint_revision": inference_REVISION,
        "runtime_versions": inference_runtime_versions(),
        "device": str(device),
        "settings": {
            "memory_policy": "exact",
            "context_cap": None,
            "context_start": 2018,
            "pruning_threshold": 0.75,
            "categorical_columns": [],
            "inference_config": "installed NoriRegressor default",
            "augmentations": ["yj"],
            "yj_skew_threshold": 10.0,
            "quantile_collapse": "mean",
        },
        "implementation_sha256": inference_hashlib.sha256(
            (
                inference_inspect.getsource(inference_predict_week)
                + inference_inspect.getsource(inference_run_blog_predictions)
            ).encode()
        ).hexdigest(),
    }


def inference_read_prediction_cache(path, provenance, query, train_rows, columns):
    """Return None for missing/stale/corrupt caches; never trust an unmanifested file."""
    path = inference_Path(path)
    manifest_path = path.with_suffix(".manifest.json")
    if not path.exists() or not manifest_path.exists():
        return None
    try:
        manifest = inference_json.loads(manifest_path.read_text())
        if manifest.get("provenance") != provenance:
            return None
        if (
            manifest.get("prediction_sha256")
            != inference_hashlib.sha256(path.read_bytes()).hexdigest()
        ):
            return None
        prediction = inference_pl.read_parquet(path)
        if not prediction.select(inference_IDENTITY_COLUMNS).equals(
            query.select(inference_IDENTITY_COLUMNS)
        ):
            return None
        if prediction["context_rows"].to_list() != [train_rows] * query.height:
            return None
        if (
            prediction["selected_feature_columns_json"].to_list()
            != [inference_json.dumps(columns)] * query.height
        ):
            return None
        required = {
            "nori_mean",
            "nori_p10",
            "nori_median",
            "nori_p90",
            "nori_quantile_taus",
            "nori_quantile_values",
        }
        if not required.issubset(prediction.columns):
            return None
        return prediction
    except (OSError, ValueError, KeyError, inference_pl.exceptions.PolarsError):
        return None


def inference_write_prediction_cache(path, prediction, provenance):
    path = inference_Path(path)
    prediction.write_parquet(path)
    path.with_suffix(".manifest.json").write_text(
        inference_json.dumps(
            {
                "provenance": provenance,
                "prediction_sha256": inference_hashlib.sha256(
                    path.read_bytes()
                ).hexdigest(),
            },
            indent=2,
        )
    )


def inference_checkpoint_path():
    path = inference_Path(
        inference_hf_hub_download(
            "Synthefy/Nori", "nori.pt", revision=inference_REVISION
        )
    )
    if (
        inference_hashlib.sha256(path.read_bytes()).hexdigest()
        != inference_CHECKPOINT_SHA256
    ):
        raise ValueError("Downloaded checkpoint does not match the blog weights")
    return path


def inference_pearson(x, y):
    """Pairwise-complete correlation, matching the original training-only selector."""
    valid = inference_np.isfinite(x) & inference_np.isfinite(y)
    if valid.sum() < 2:
        return None
    x, y = (x[valid], y[valid])
    x, y = (x - x.mean(), y - y.mean())
    xx, yy = (float(inference_np.dot(x, x)), float(inference_np.dot(y, y)))
    if xx <= 0 or yy <= 0:
        return None
    return float(
        inference_np.clip(inference_np.dot(x, y) / inference_np.sqrt(xx * yy), -1, 1)
    )


def inference_select_columns(train, candidates, threshold=0.75):
    """Rank by absolute target correlation; alphabetical ties; drop correlation > .75."""
    if len(candidates) != len(set(candidates)) or not candidates:
        raise ValueError("Candidate columns must be nonempty and unique")
    target = train[inference_TARGET].to_numpy().astype(inference_np.float64)
    values = {c: train[c].to_numpy().astype(inference_np.float64) for c in candidates}
    correlations = {c: inference_pearson(values[c], target) for c in sorted(candidates)}
    scores = {c: None if r is None else abs(r) for c, r in correlations.items()}
    ranked = sorted(candidates, key=lambda c: (scores[c] is None, -(scores[c] or 0), c))
    kept = []
    for c in ranked:
        if scores[c] is None:
            continue
        redundant = False
        for other in kept:
            r = inference_pearson(values[c], values[other])
            if r is not None and abs(r) > threshold:
                redundant = True
        if not redundant:
            kept.append(c)
    if not kept:
        raise ValueError("No nonconstant features remain")
    return kept


def inference_modeling_rows(rows, season=2025):
    return rows.filter(
        inference_pl.col("live_evaluation_eligible")
        & inference_pl.col("season").is_between(2018, season)
        & inference_pl.col(inference_TARGET).is_not_null()
        & inference_pl.col(inference_TARGET).is_finite()
        & inference_pl.col("official_passing_yards").is_not_null()
        & inference_pl.col("official_passing_yards").is_finite()
    ).sort(["season", "week", "live_decision_utc", "game_id", "team"])

In [ ]:
from __future__ import annotations


def inference_predict_week(estimator, train, query, columns):
    x = train.select(columns).to_numpy().astype(inference_np.float32, copy=False)
    xq = query.select(columns).to_numpy().astype(inference_np.float32, copy=False)
    if inference_np.isinf(x).any() or inference_np.isinf(xq).any():
        raise ValueError("Infinite feature values")
    estimator.fit(
        x, train[inference_TARGET].to_numpy().astype(inference_np.float64, copy=False)
    )
    distribution = estimator.predict(xq, output_type="full")
    report = getattr(estimator, "memory_report_", None) or {}
    if report.get("dropped_context_rows", 0):
        raise RuntimeError("Exact reproduction cannot drop context rows")
    bank = inference_np.asarray(distribution["quantiles"], dtype=inference_np.float64)
    taus = inference_np.asarray(distribution["taus"], dtype=inference_np.float64)
    if bank.ndim != 2:
        raise ValueError("Unexpected quantile bank dimensions")
    if bank.shape[0] != query.height and bank.shape[1] == query.height:
        bank = bank.T
    if bank.shape != (query.height, taus.size):
        raise ValueError("Unexpected quantile bank shape")
    current = query["live_qb_passing_yards"].to_numpy().astype(inference_np.float64)
    final = bank + current[:, None]
    quantiles = inference_np.asarray(
        [[inference_np.interp(t, taus, row) for t in (0.1, 0.5, 0.9)] for row in final]
    )
    return query.select(inference_IDENTITY_COLUMNS).with_columns(
        inference_pl.lit(train.height, dtype=inference_pl.Int64).alias("context_rows"),
        inference_pl.lit(len(columns), dtype=inference_pl.Int64).alias(
            "selected_feature_count"
        ),
        inference_pl.lit(inference_json.dumps(columns)).alias(
            "selected_feature_columns_json"
        ),
        inference_pl.Series(
            "nori_mean",
            inference_np.asarray(distribution["mean"], dtype=inference_np.float64)
            + current,
        ),
        inference_pl.Series("nori_p10", quantiles[:, 0]),
        inference_pl.Series("nori_median", quantiles[:, 1]),
        inference_pl.Series("nori_p90", quantiles[:, 2]),
        inference_pl.Series("nori_quantile_taus", [taus.tolist()] * query.height),
        inference_pl.Series("nori_quantile_values", final.tolist()),
    )


def inference_prediction_metrics(frame):
    actual = frame["official_passing_yards"].to_numpy()
    residual = actual - frame["nori_median"].to_numpy()
    pinball = {}
    for tau, column in ((0.1, "nori_p10"), (0.5, "nori_median"), (0.9, "nori_p90")):
        error = actual - frame[column].to_numpy()
        pinball[str(tau)] = float(
            inference_np.maximum(tau * error, (tau - 1) * error).mean()
        )
    return {
        "rows": frame.height,
        "mae": float(inference_np.abs(residual).mean()),
        "rmse": float(inference_np.sqrt(inference_np.square(residual).mean())),
        "pinball_loss": pinball,
        "mean_pinball_loss": float(inference_np.mean(list(pinball.values()))),
        "p10_p90_coverage": float(
            (
                (actual >= frame["nori_p10"].to_numpy())
                & (actual <= frame["nori_p90"].to_numpy())
            ).mean()
        ),
    }


def inference_run_blog_predictions(
    q1_rows,
    ht_rows,
    output_dir,
    weeks=tuple(range(1, 19)),
    device="cpu",
    season=2025,
    horizons=("q1", "halftime"),
    resume=False,
):
    """Compute checkpoints without a context cap; optionally resume verified local work.

    ``weeks`` limits scoring origins, not historical context. CPU execution of
    every week may take hours; choose a single week for a first smoke run.
    ``resume=True`` only reuses matching locally computed, fingerprinted results.
    """
    weeks = tuple(weeks)
    if not weeks or len(set(weeks)) != len(weeks):
        raise ValueError("weeks must be nonempty and unique")
    if (
        not horizons
        or len(set(horizons)) != len(horizons)
        or set(horizons) - {"q1", "halftime"}
    ):
        raise ValueError("horizons must contain q1 and/or halftime, without duplicates")
    directory = inference_Path(output_dir)
    directory.mkdir(parents=True, exist_ok=True)
    model = inference_NoriRegressor(
        model_path=str(inference_checkpoint_path()),
        device=device,
        memory_policy="exact",
    )
    candidates = pipeline_blog_feature_columns()
    outputs = {}
    for horizon, raw in (("q1", q1_rows), ("halftime", ht_rows)):
        if horizon not in horizons:
            continue
        rows = inference_modeling_rows(raw, season)
        predictions, selections = ([], {})
        for week in sorted(weeks):
            train = rows.filter(
                (inference_pl.col("season") < season)
                | (inference_pl.col("season") == season)
                & (inference_pl.col("week") < week)
            )
            query = rows.filter(
                (inference_pl.col("season") == season)
                & (inference_pl.col("week") == week)
            )
            if train.is_empty() or query.is_empty():
                raise ValueError(f"Empty context/query for {horizon} week {week}")
            columns = inference_select_columns(train, candidates)
            print(
                f"{horizon} week {week}: {train.height} context, {query.height} queries, {len(columns)} features",
                flush=True,
            )
            path = directory / f"{horizon}_{season}_week{week}_predictions.parquet"
            provenance = inference_prediction_provenance(train, query, columns, device)
            prediction = (
                inference_read_prediction_cache(
                    path, provenance, query, train.height, columns
                )
                if resume
                else None
            )
            if prediction is None:
                prediction = inference_predict_week(model, train, query, columns)
                inference_write_prediction_cache(path, prediction, provenance)
            else:
                print(f"Resumed verified {horizon} week {week}", flush=True)
            predictions.append(prediction)
            selections[week] = columns
        frame = inference_pl.concat(predictions).sort(
            ["week", "live_decision_utc", "game_id", "team"]
        )
        frame.write_parquet(directory / f"{horizon}_{season}_predictions.parquet")
        (directory / f"{horizon}_{season}_manifest.json").write_text(
            inference_json.dumps(
                {
                    "checkpoint_revision": inference_REVISION,
                    "checkpoint_sha256": inference_CHECKPOINT_SHA256,
                    "runtime_versions": {
                        name: inference_version(name)
                        for name in ("synthefy-nori", "numpy", "torch", "polars")
                    },
                    "device": device,
                    "context_start": 2018,
                    "context_cap": None,
                    "categorical_columns": [],
                    "pruning_threshold": 0.75,
                    "selected_columns": selections,
                    "metrics": inference_prediction_metrics(frame),
                },
                indent=2,
            )
        )
        outputs[horizon] = frame
    return outputs

### Public Kalshi request caching

In [ ]:
from __future__ import annotations

import hashlib as markets_hashlib

import json as markets_json

from pathlib import Path as markets_Path

from urllib.parse import quote as markets_quote

import pandas as markets_pd

import requests as markets_requests

from requests.adapters import HTTPAdapter as markets_HTTPAdapter

from urllib3.util.retry import Retry as markets_Retry

markets_BASE_URL = "https://external-api.kalshi.com/trade-api/v2"


class markets_PublicKalshi:
    """Unauthenticated GET-only client; immutable request-keyed provenance cache."""

    def __init__(self, cache_dir):
        self.cache_dir = markets_Path(cache_dir) / "kalshi"
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.session = markets_requests.Session()
        self.session.mount(
            "https://",
            markets_HTTPAdapter(
                max_retries=markets_Retry(
                    total=5,
                    backoff_factor=1,
                    status_forcelist=[429, 500, 502, 503, 504],
                    allowed_methods=["GET"],
                )
            ),
        )

    def get(self, endpoint, **params):
        url = f"{markets_BASE_URL}/{endpoint}"
        identity = markets_json.dumps([url, params], sort_keys=True)
        path = self.cache_dir / (
            markets_hashlib.sha256(identity.encode()).hexdigest() + ".json"
        )
        if path.exists():
            record = markets_json.loads(path.read_text())
            if (
                markets_hashlib.sha256(
                    markets_json.dumps(record["payload"], sort_keys=True).encode()
                ).hexdigest()
                != record["payload_sha256"]
            ):
                raise ValueError(f"Corrupt cached payload: {path}")
            return record["payload"]
        response = self.session.get(url, params=params, timeout=60)
        response.raise_for_status()
        payload = response.json()
        record = dict(
            url=response.url,
            retrieved_utc=markets_pd.Timestamp.now(tz="UTC").isoformat(),
            sha256=markets_hashlib.sha256(response.content).hexdigest(),
            payload_sha256=markets_hashlib.sha256(
                markets_json.dumps(payload, sort_keys=True).encode()
            ).hexdigest(),
            payload=payload,
        )
        temporary = path.with_suffix(".tmp")
        temporary.write_text(markets_json.dumps(record))
        temporary.replace(path)
        return payload

    def markets(self):
        """Download all archived passing-yard markets, following every cursor."""
        result, cursor, seen = ([], "", set())
        while True:
            params = {"series_ticker": "KXNFLPASSYDS", "limit": 1000}
            if cursor:
                params["cursor"] = cursor
            page = self.get("historical/markets", **params)
            result.extend(page["markets"])
            cursor = page.get("cursor", "")
            if not cursor:
                break
            if cursor in seen:
                raise RuntimeError(
                    "Repeated historical-market cursor; download is incomplete"
                )
            seen.add(cursor)
        return list({m["ticker"]: m for m in result}.values())

    def quote_at(self, ticker, decision_utc, max_age_seconds=300):
        """Last completed minute candle at/before decision; never use future quotes."""
        decision = markets_pd.Timestamp(decision_utc)
        if markets_pd.isna(decision) or decision.tzinfo is None:
            raise ValueError("Decision timestamp must be known and timezone-aware")
        end = int(decision.timestamp())
        payload = self.get(
            f"historical/markets/{markets_quote(ticker, safe='')}/candlesticks",
            start_ts=end - max_age_seconds - 60,
            end_ts=end,
            period_interval=1,
        )
        candles = [
            c
            for c in payload["candlesticks"]
            if 0 <= decision.timestamp() - c["end_period_ts"] <= max_age_seconds
        ]
        if not candles:
            return {}
        candle = max(candles, key=lambda c: c["end_period_ts"])

        def close(field):
            values = candle.get(field) or {}
            if values.get("close_dollars") is not None:
                return float(values["close_dollars"])
            value = values.get("close")
            if value is None:
                return None
            return float(value) if isinstance(value, str) else float(value) / 100

        return dict(
            quote_utc=markets_pd.Timestamp(
                candle["end_period_ts"], unit="s", tz="UTC"
            ).isoformat(),
            quote_age_seconds=decision.timestamp() - candle["end_period_ts"],
            yes_bid=close("yes_bid"),
            yes_ask=close("yes_ask"),
        )

### Helpers for the one blog strategy

These functions match markets, calculate fees and probabilities, and apply the two checkpoints of the same rule. `run_blog_backtest` is the only strategy entry point: try Q1 first; only if it passes, consider a probability-confirmed halftime bet.

In [ ]:
from __future__ import annotations

import math as strategy_math

import re as strategy_re

from datetime import timezone as strategy_timezone

from datetime import datetime as strategy_datetime

from datetime import timedelta as strategy_timedelta

from decimal import Decimal as strategy_Decimal

from pathlib import Path as strategy_Path

from typing import Any as strategy_Any

from urllib.parse import quote as strategy_quote

import numpy as strategy_np

import polars as strategy_pl

import requests as strategy_requests

strategy_UTC = strategy_timezone.utc

strategy_EDGE_THRESHOLD = 0.1

strategy_NUMERIC_TOLERANCE = 1e-12

strategy_KEYS = ["game_id", "team"]

strategy_IDENTITY_COLUMNS = (
    "week",
    "game_id",
    "team",
    "opponent_team",
    "actual_qb_id",
    "actual_qb_name",
    "official_passing_yards",
)

strategy_SELECTED_COLUMNS = (
    "live_anchor_utc",
    "live_decision_utc",
    "live_qb_passing_yards",
    "nori_median",
    "ticker",
    "line",
    "side",
    "model_probability",
    "yes_bid",
    "yes_ask",
    "spread",
    "quote_ts",
    "quote_age_seconds",
    "entry_price",
    "fee",
    "expected_net_edge",
    "settlement_value",
    "capital",
    "profit",
)


def strategy__parse_time(value: str) -> strategy_datetime:
    return strategy_datetime.fromisoformat(value).astimezone(strategy_UTC)


def strategy__as_datetime(value: object) -> strategy_datetime:
    if isinstance(value, strategy_datetime):
        return value.astimezone(strategy_UTC)
    return strategy__parse_time(str(value))


def strategy_normalize_name(value: str) -> str:
    return strategy_re.sub("[^a-z0-9]", "", value.lower())


def strategy_market_player_name(title: str) -> str:
    return strategy_re.split(
        "\\s+records\\s+\\d+\\+|:\\s*\\d+\\+",
        title,
        maxsplit=1,
        flags=strategy_re.IGNORECASE,
    )[0].strip()


def strategy_kalshi_taker_fee(price: float, contracts: int, rate: float) -> float:
    raw_fee = rate * contracts * price * (1.0 - price)
    return strategy_math.ceil(raw_fee * 100.0 - 1e-12) / 100.0


def strategy_bid_ask_spread(bid: float, ask: float) -> float:
    """Respect exact decimal price cutoffs (0.55 - 0.45 is ten cents)."""
    return float(strategy_Decimal(str(ask)) - strategy_Decimal(str(bid)))


def strategy__close_value(side: dict[str, strategy_Any] | None) -> float | None:
    if not side:
        return None
    if side.get("close_dollars") is not None:
        return float(side["close_dollars"])
    value = side.get("close")
    if value is None:
        return None
    return float(value) if isinstance(value, str) else float(value) / 100.0


def strategy__settlement_value(market: dict[str, strategy_Any]) -> float:
    value = market.get("settlement_value_dollars")
    if value is not None:
        return float(value)
    if market.get("result") == "yes":
        return 1.0
    if market.get("result") == "no":
        return 0.0
    raise ValueError(f"Market {market['ticker']} lacks a final settlement value")


def strategy_match_live_markets_to_predictions(
    markets: list[dict[str, strategy_Any]], predictions: strategy_pl.DataFrame
) -> tuple[list[dict[str, strategy_Any]], dict[str, int]]:
    required = {
        "game_id",
        "week",
        "kickoff_utc",
        "live_anchor_utc",
        "live_decision_utc",
        "team",
        "opponent_team",
        "actual_qb_id",
        "actual_qb_name",
    }
    missing = sorted(required.difference(predictions.columns))
    if missing:
        raise ValueError(
            f"live predictions are missing market-match columns: {missing}"
        )
    by_name: dict[str, list[dict[str, strategy_Any]]] = {}
    for row in predictions.select(*sorted(required)).to_dicts():
        by_name.setdefault(
            strategy_normalize_name(str(row["actual_qb_name"])), []
        ).append(row)
    matched: list[dict[str, strategy_Any]] = []
    missing_name = 0
    ambiguous = 0
    for market in markets:
        if market.get("floor_strike") is None or not market.get(
            "expected_expiration_time"
        ):
            continue
        candidates = by_name.get(
            strategy_normalize_name(
                strategy_market_player_name(str(market.get("title", "")))
            ),
            [],
        )
        expiration = strategy__parse_time(str(market["expected_expiration_time"]))
        candidates = [
            row
            for row in candidates
            if strategy_timedelta(hours=1)
            <= expiration - strategy__as_datetime(row["kickoff_utc"])
            <= strategy_timedelta(hours=8)
        ]
        if not candidates:
            missing_name += 1
            continue
        if len(candidates) != 1:
            ambiguous += 1
            continue
        matched.append(
            {**market, **candidates[0], "model_line": float(market["floor_strike"])}
        )
    return (
        matched,
        {
            "input_markets": len(markets),
            "matched_markets": len(matched),
            "unmatched_market_rows": missing_name,
            "ambiguous_market_rows": ambiguous,
        },
    )


def strategy_probability_over_line(
    taus: list[float], quantile_values: list[float], line: float
) -> float:
    levels = strategy_np.asarray(taus, dtype=strategy_np.float64)
    values = strategy_np.asarray(quantile_values, dtype=strategy_np.float64)
    if (
        levels.ndim != 1
        or values.ndim != 1
        or levels.size != values.size
        or (levels.size < 2)
    ):
        raise ValueError("invalid predictive quantile row")
    if not strategy_np.isfinite(levels).all() or not strategy_np.isfinite(values).all():
        raise ValueError("predictive quantile row contains non-finite values")
    values = strategy_np.maximum.accumulate(values)
    probability = 1.0 - strategy_np.interp(line, values, levels, left=0.0, right=1.0)
    return float(strategy_np.clip(probability, 0.0, 1.0))


def strategy__empty_decision(
    prediction: dict[str, strategy_Any], status: str
) -> dict[str, strategy_Any]:
    return {
        "game_id": prediction["game_id"],
        "week": prediction["week"],
        "team": prediction["team"],
        "opponent_team": prediction["opponent_team"],
        "actual_qb_id": prediction["actual_qb_id"],
        "actual_qb_name": prediction["actual_qb_name"],
        "live_anchor_utc": prediction["live_anchor_utc"],
        "live_decision_utc": prediction["live_decision_utc"],
        "live_qb_passing_yards": prediction["live_qb_passing_yards"],
        "official_passing_yards": prediction["official_passing_yards"],
        "nori_median": prediction["nori_median"],
        "ticker": None,
        "line": None,
        "side": None,
        "model_probability": None,
        "yes_bid": None,
        "yes_ask": None,
        "spread": None,
        "quote_ts": None,
        "quote_age_seconds": None,
        "entry_price": None,
        "fee": None,
        "expected_net_edge": None,
        "settlement_value": None,
        "capital": None,
        "profit": None,
        "bet_taken": False,
        "decision_status": status,
        "realized_profit": None,
    }

In [ ]:
from __future__ import annotations


def _select_checkpoint_candidates(
    matched_markets: list[dict[str, strategy_Any]],
    predictions: strategy_pl.DataFrame,
    quotes: dict[str, dict[str, strategy_Any]],
    checkpoint: str,
) -> strategy_pl.DataFrame:
    """Find candidates for one checkpoint of the fixed blog rule, not a separate strategy."""
    if checkpoint not in {"q1", "halftime"}:
        raise ValueError(f"unsupported checkpoint: {checkpoint}")
    prediction_map = {
        (str(row["game_id"]), str(row["actual_qb_id"])): row
        for row in predictions.to_dicts()
    }
    available_keys: set[tuple[str, str]] = set()
    quoted_keys: set[tuple[str, str]] = set()
    fresh_keys: set[tuple[str, str]] = set()
    valid_spread_keys: set[tuple[str, str]] = set()
    candidates: list[dict[str, strategy_Any]] = []
    available_markets = 0
    quoted_markets = 0
    fresh_markets = 0
    for market in matched_markets:
        key = (str(market["game_id"]), str(market["actual_qb_id"]))
        prediction = prediction_map.get(key)
        if prediction is None:
            continue
        decision = strategy__as_datetime(prediction["live_decision_utc"])
        created = strategy__parse_time(str(market["created_time"]))
        open_time = strategy__parse_time(str(market["open_time"]))
        close_time = strategy__parse_time(str(market["close_time"]))
        if created > decision or open_time > decision or close_time <= decision:
            continue
        available_keys.add(key)
        available_markets += 1
        quote = quotes.get(str(market["ticker"]), {})
        quote_ts = quote.get("quote_ts")
        yes_bid = quote.get("yes_bid")
        yes_ask = quote.get("yes_ask")
        if quote_ts is None or yes_bid is None or yes_ask is None:
            continue
        quoted_keys.add(key)
        quoted_markets += 1
        quote_time = strategy_datetime.fromtimestamp(int(quote_ts), tz=strategy_UTC)
        anchor = strategy__as_datetime(prediction["live_anchor_utc"])
        quote_age_seconds = (decision - quote_time).total_seconds()
        if (
            quote_time < anchor
            or quote_time > decision
            or quote_age_seconds > 5 * 60
        ):
            continue
        fresh_keys.add(key)
        fresh_markets += 1
        spread = strategy_bid_ask_spread(float(yes_bid), float(yes_ask))
        if (
            not 0.0 <= float(yes_bid) <= float(yes_ask) <= 1.0
            or spread > 0.10
        ):
            continue
        valid_spread_keys.add(key)
        probability_yes = strategy_probability_over_line(
            prediction["nori_quantile_taus"],
            prediction["nori_quantile_values"],
            float(market["model_line"]),
        )
        settlement_yes = strategy__settlement_value(market)
        for side, probability, entry_price, settlement in (
            ("yes", probability_yes, float(yes_ask), settlement_yes),
            ("no", 1.0 - probability_yes, 1.0 - float(yes_bid), 1.0 - settlement_yes),
        ):
            fee = strategy_kalshi_taker_fee(
                entry_price, 1, 0.07
            )
            expected_net_edge = (
                probability - entry_price - fee
            )
            capital = entry_price + fee
            profit = settlement - capital
            candidates.append(
                {
                    "game_id": prediction["game_id"],
                    "week": prediction["week"],
                    "team": prediction["team"],
                    "opponent_team": prediction["opponent_team"],
                    "actual_qb_id": prediction["actual_qb_id"],
                    "actual_qb_name": prediction["actual_qb_name"],
                    "live_anchor_utc": prediction["live_anchor_utc"],
                    "live_decision_utc": prediction["live_decision_utc"],
                    "live_qb_passing_yards": prediction["live_qb_passing_yards"],
                    "official_passing_yards": prediction["official_passing_yards"],
                    "nori_median": prediction["nori_median"],
                    "ticker": market["ticker"],
                    "line": float(market["model_line"]) + 0.5,
                    "side": side,
                    "model_probability": probability,
                    "yes_bid": float(yes_bid),
                    "yes_ask": float(yes_ask),
                    "spread": spread,
                    "quote_ts": int(quote_ts),
                    "quote_age_seconds": quote_age_seconds,
                    "entry_price": entry_price,
                    "fee": fee,
                    "expected_net_edge": expected_net_edge,
                    "settlement_value": settlement,
                    "capital": capital,
                    "profit": profit,
                }
            )

    def candidate_priority(row: dict[str, strategy_Any]) -> tuple:
        prefix = (row["game_id"], row["actual_qb_id"])
        if checkpoint == "q1":
            return prefix + (
                round(abs((row["yes_bid"] + row["yes_ask"]) / 2.0 - 0.5), 10),
                row["ticker"],
                -row["expected_net_edge"],
                row["side"],
            )
        return (
            row["game_id"],
            row["actual_qb_id"],
            -row["expected_net_edge"],
            row["ticker"],
            row["side"],
        )

    candidates.sort(key=candidate_priority)
    best_by_key: dict[tuple[str, str], dict[str, strategy_Any]] = {}
    for candidate in candidates:
        key = (str(candidate["game_id"]), str(candidate["actual_qb_id"]))
        best_by_key.setdefault(key, candidate)
    decisions: list[dict[str, strategy_Any]] = []
    for key, prediction in prediction_map.items():
        if key in best_by_key:
            decision = best_by_key[key]
            decision["bet_taken"] = (
                decision["expected_net_edge"] >= strategy_EDGE_THRESHOLD
            )
            decision["decision_status"] = "bet" if decision["bet_taken"] else "pass"
            decision["realized_profit"] = (
                decision["profit"] if decision["bet_taken"] else 0.0
            )
        elif key not in available_keys:
            decision = strategy__empty_decision(prediction, "no_live_market")
        elif key not in quoted_keys:
            decision = strategy__empty_decision(prediction, "no_post_anchor_quote")
        elif key not in fresh_keys:
            decision = strategy__empty_decision(prediction, "stale_quote")
        elif key not in valid_spread_keys:
            decision = strategy__empty_decision(prediction, "spread_too_wide")
        else:
            decision = strategy__empty_decision(prediction, "no_eligible_candidate")
        decisions.append(decision)
    decisions.sort(key=lambda row: (row["week"], row["game_id"], row["team"]))
    decision_frame = strategy_pl.DataFrame(decisions, infer_schema_length=None)
    return decision_frame


def strategy__float(value: object, label: str) -> float:
    result = float(value)
    if not strategy_math.isfinite(result):
        raise ValueError(f"{label} must be finite")
    return result


def strategy_q1_yes_probability(
    prediction: dict[str, strategy_Any], integer_line: float
) -> float:
    return strategy_probability_over_line(
        prediction["nori_quantile_taus"],
        prediction["nori_quantile_values"],
        integer_line - 0.5,
    )

In [ ]:
from __future__ import annotations


def _apply_q1_priority_and_halftime_confirmation(
    q1: strategy_pl.DataFrame,
    halftime: strategy_pl.DataFrame,
    q1_predictions: strategy_pl.DataFrame,
) -> strategy_pl.DataFrame:
    """Apply Q1 priority and inclusive same-strike probability confirmation."""
    q1 = q1.sort(strategy_KEYS)
    halftime = halftime.sort(strategy_KEYS)
    q1_predictions = q1_predictions.sort(strategy_KEYS)
    if not q1.select(strategy_KEYS).equals(
        halftime.select(strategy_KEYS)
    ) or not q1.select(strategy_KEYS).equals(q1_predictions.select(strategy_KEYS)):
        raise ValueError("probability-confirmation sources are not aligned")
    records: list[dict[str, strategy_Any]] = []
    for q1_row, half_row, prediction in zip(
        q1.to_dicts(), halftime.to_dicts(), q1_predictions.to_dicts(), strict=True
    ):
        q1_eligible = bool(q1_row["bet_taken"])
        half_cost_eligible = (
            bool(half_row["bet_taken"])
            and strategy__float(half_row["expected_net_edge"], "halftime expected edge")
            >= strategy_EDGE_THRESHOLD - strategy_NUMERIC_TOLERANCE
        )
        halftime_reached = not q1_eligible
        q1_side_probability = None
        halftime_side_probability = None
        probability_change = None
        probability_confirmed = None
        confirmation_ticker = None
        confirmation_line = None
        confirmation_side = None
        if q1_eligible:
            selected = q1_row
            selected_horizon = "q1"
            status = "bet"
        elif half_cost_eligible:
            confirmation_ticker = str(half_row["ticker"])
            confirmation_line = strategy__float(half_row["line"], "halftime line")
            confirmation_side = str(half_row["side"])
            q1_probability_yes = strategy_q1_yes_probability(
                prediction, confirmation_line
            )
            if confirmation_side == "yes":
                q1_side_probability = q1_probability_yes
            elif confirmation_side == "no":
                q1_side_probability = 1.0 - q1_probability_yes
            else:
                raise ValueError("probability-confirmation source has an invalid side")
            halftime_side_probability = strategy__float(
                half_row["model_probability"], "halftime side probability"
            )
            probability_change = halftime_side_probability - q1_side_probability
            probability_confirmed = (
                halftime_side_probability + strategy_NUMERIC_TOLERANCE
                >= q1_side_probability
            )
            if probability_confirmed:
                selected = half_row
                selected_horizon = "halftime"
                status = "bet"
            else:
                selected = None
                selected_horizon = "abstain"
                status = "pass_probability_confirmation"
        else:
            selected = None
            selected_horizon = "abstain"
            status = "abstain"
        record = {column: q1_row[column] for column in strategy_IDENTITY_COLUMNS}
        record.update(
            {
                "q1_eligible": q1_eligible,
                "halftime_cost_eligible": half_cost_eligible,
                "halftime_reached": halftime_reached,
                "probability_confirmed": probability_confirmed,
                "q1_selected_side_probability": q1_side_probability,
                "halftime_selected_side_probability": halftime_side_probability,
                "selected_side_probability_change": probability_change,
                "confirmation_ticker": confirmation_ticker,
                "confirmation_line": confirmation_line,
                "confirmation_side": confirmation_side,
                "selected_horizon": selected_horizon,
                "bet_taken": selected is not None,
                "decision_status": status,
            }
        )
        for column in strategy_SELECTED_COLUMNS:
            record[column] = selected[column] if selected is not None else None
        record["realized_profit"] = selected["profit"] if selected is not None else None
        records.append(record)
    decisions = strategy_pl.DataFrame(records, infer_schema_length=None).sort(
        strategy_KEYS
    )
    return decisions


def strategy_quote_after_anchor(client, market):
    """Same anchor-to-decision minute-close request as the original experiment."""
    anchor = int(strategy__as_datetime(market["live_anchor_utc"]).timestamp())
    decision = int(strategy__as_datetime(market["live_decision_utc"]).timestamp())
    payload = client.get(
        f"historical/markets/{strategy_quote(market['ticker'], safe='')}/candlesticks",
        start_ts=anchor,
        end_ts=decision,
        period_interval=1,
    )
    candles = [
        c
        for c in payload["candlesticks"]
        if anchor <= int(c["end_period_ts"]) <= decision
    ]
    result = dict(
        ticker=market["ticker"],
        anchor_ts=anchor,
        decision_ts=decision,
        quote_ts=None,
        yes_bid=None,
        yes_ask=None,
    )
    if candles:
        candle = max(candles, key=lambda c: int(c["end_period_ts"]))
        result.update(
            quote_ts=int(candle["end_period_ts"]),
            yes_bid=strategy__close_value(candle.get("yes_bid")),
            yes_ask=strategy__close_value(candle.get("yes_ask")),
        )
    return result


def strategy_summarize(decisions, execution_cost=0.05):
    if not strategy_math.isfinite(execution_cost) or execution_cost < 0:
        raise ValueError("Execution allowance must be finite and nonnegative")
    trades = decisions.filter(strategy_pl.col("bet_taken")).to_dicts()
    capital = sum((r["capital"] + execution_cost for r in trades))
    pnl = sum((r["profit"] - execution_cost for r in trades))
    return {
        "status": "quote_based_simulation",
        "bets": len(trades),
        "games": len({r["game_id"] for r in trades}),
        "wins": sum((r["settlement_value"] == 1 for r in trades)),
        "capital": capital,
        "pnl": pnl,
        "roi": pnl / capital if capital else None,
        "execution_cost": execution_cost,
        "limitation": "Recorded quotes do not establish available size or fills. 2025 was used for strategy exploration.",
    }

In [ ]:
from __future__ import annotations


def _replay_blog_quotes(
    q1, halftime, markets, quotes_by_horizon, execution_cost=0.05
):
    """Original selections: Q1 balanced market, then confirmed max-edge halftime.

    Input quantiles predict final passing yards (not remaining yards). No future
    transaction price or settlement enters the candidate-ranking rule. Quotes
    are raw public responses transformed by quote_after_anchor.
    """
    if q1.is_empty() or halftime.is_empty():
        raise ValueError("Both checkpoints need predictions")
    for predictions in (q1, halftime):
        if predictions.select(strategy_KEYS).is_duplicated().any():
            raise ValueError("Duplicate QB-game prediction keys")
    decisions = {}
    coverage = {}
    for horizon, predictions in (("q1", q1), ("halftime", halftime)):
        matched, counts = strategy_match_live_markets_to_predictions(
            markets, predictions
        )
        decisions[horizon] = _select_checkpoint_candidates(
            matched,
            predictions,
            quotes_by_horizon[horizon],
            checkpoint=horizon,
        )
        coverage[horizon] = counts
    combined = _apply_q1_priority_and_halftime_confirmation(
        decisions["q1"], decisions["halftime"], q1
    )
    combined = combined.with_columns(
        strategy_pl.when(strategy_pl.col("bet_taken"))
        .then(strategy_pl.col("capital") + execution_cost)
        .otherwise(None)
        .alias("stressed_capital"),
        strategy_pl.when(strategy_pl.col("bet_taken"))
        .then(strategy_pl.col("profit") - execution_cost)
        .otherwise(None)
        .alias("stressed_profit"),
    )
    report = strategy_summarize(combined, execution_cost)
    report["coverage"] = coverage
    report["checkpoints"] = {
        h: strategy_summarize(
            combined.filter(strategy_pl.col("selected_horizon") == h), execution_cost
        )
        for h in ("q1", "halftime")
    }
    return (combined, report)


def run_blog_backtest(q1, halftime, cache_dir, execution_cost=0.05):
    """Fetch public archives; network unavailability never invalidates forecasts.

    A failed page/quote aborts only the optional market stage. Do not report ROI
    from an incomplete download. Software/data-contract errors remain visible.
    """
    client = markets_PublicKalshi(strategy_Path(cache_dir))
    try:
        markets = client.markets()
        quotes = {}
        for horizon, predictions in (("q1", q1), ("halftime", halftime)):
            matched, _ = strategy_match_live_markets_to_predictions(
                markets, predictions
            )
            quotes[horizon] = {}
            for index, market in enumerate(matched):
                quotes[horizon][market["ticker"]] = strategy_quote_after_anchor(
                    client, market
                )
                if (index + 1) % 100 == 0:
                    print(
                        f"Kalshi {horizon}: {index + 1}/{len(matched)} quotes",
                        flush=True,
                    )
        return _replay_blog_quotes(
            q1, halftime, markets, quotes, execution_cost
        )
    except (
        strategy_requests.RequestException,
        FileNotFoundError,
        TimeoutError,
    ) as error:
        return (
            strategy_pl.DataFrame(),
            {
                "status": "unavailable",
                "reason": str(error),
                "roi": None,
                "note": "Predictions are retained. Rerun this optional stage when public history is available.",
            },
        )

In [ ]:
build_blog_features = pipeline_build_blog_features
blog_feature_columns = pipeline_blog_feature_columns
run_blog_predictions = inference_run_blog_predictions
prediction_metrics = inference_prediction_metrics
modeling_rows = inference_modeling_rows
probability_over_line = strategy_probability_over_line

## 3. Pull football data and build the two checkpoint tables
nflreadpy downloads schedules, player statistics, depth charts, participation,
play-by-play, and team metadata. Full-game rolling form uses 2016–2017 as
warmup. Model context starts in 2018 and expands only with earlier weeks.

The 140 numeric candidates include QB form, offense, defense, game/season
context, game state at the checkpoint, and the QB's matching Q1/halftime
history. The original selected model uses identity to compute individual
history; it does **not** feed categorical QB IDs or weather to Nori.

Q1 uses the earliest timestamped Q2 record as its boundary, excluding that
record. Halftime uses the last Q2 record. The decision is two minutes later.
Finalized play data may include corrections and do not prove feed arrival time.

In [ ]:
features = build_blog_features(CACHE_DIR, OUTPUT_DIR, refresh=REFRESH_SOURCE_DATA)
overview = pd.DataFrame(
    [
        {
            "checkpoint": h,
            "rows": table.height,
            "eligible_2025_rows": modeling_rows(table)
            .filter(pl.col("season") == 2025)
            .height,
            "candidate_features": len(blog_feature_columns()),
        }
        for h, table in features.items()
    ]
)
display(overview)

In [ ]:
example_columns = [
    "game_id",
    "week",
    "actual_qb_name",
    "live_decision_utc",
    "live_qb_passing_yards",
    "live_qb_attempts",
    "official_passing_yards",
]
history_columns = [
    c for c in blog_feature_columns() if c.startswith("checkpoint_history_")
][:3]
display(
    features["q1"]
    .filter(pl.col("season") == 2025)
    .select(example_columns + history_columns)
    .head(8)
    .to_pandas()
)
print("All candidate feature names:")
print("\n".join(blog_feature_columns()))

## 4. Run Nori—fresh predictions, week by week
For each week and checkpoint, rank candidate features using correlations
computed on past context only. Drop a feature when its absolute correlation
with an already-kept feature exceeds 0.75. Use the entire retained context:
no subsampling or different lightweight model is substituted.

The pinned public Nori checkpoint predicts **remaining** yards. Adding the
yards already thrown converts every quantile into a final-yard prediction.
fit() stores the context; it does not train new model weights. Each week's
predictions and chosen columns are saved as they finish.
On a rerun, RESUME_PREDICTIONS checks local provenance fingerprints before
reusing a completed week. Changed inputs, selected features, model, runtime,
or device trigger fresh inference. Set it to False to recompute every week.

In [ ]:
predictions = run_blog_predictions(
    features["q1"],
    features["halftime"],
    OUTPUT_DIR,
    weeks=WEEKS,
    device=DEVICE,
    season=2025,
    resume=RESUME_PREDICTIONS,
)

## 5. Measure forecast error and coverage
MAE and RMSE below use the predictive median. Pinball loss measures quantile
accuracy; P10–P90 coverage is the fraction of final totals inside the predicted
middle 80%. These are forecast metrics, not betting returns.

In [ ]:
metrics = {h: prediction_metrics(frame) for h, frame in predictions.items()}
display(pd.DataFrame(metrics).T.drop(columns=["pinball_loss"]))
display(
    pd.DataFrame(
        {h: values["pinball_loss"] for h, values in metrics.items()}
    ).rename_axis("quantile")
)
(OUTPUT_DIR / "forecast_metrics.json").write_text(json.dumps(metrics, indent=2))

In [ ]:
display(
    predictions["q1"]
    .select(
        "week",
        "actual_qb_name",
        "live_qb_passing_yards",
        "nori_p10",
        "nori_median",
        "nori_p90",
        "official_passing_yards",
        "context_rows",
        "selected_feature_count",
    )
    .head(12)
    .to_pandas()
)

## 6. See the distribution in 5-yard bins
Prefer the blog's Burrow example when that week is included; otherwise use the
first predicted row. This plot is computed from this run's quantiles. Values
are not copied from the blog. Bar heights approximate probability mass by
interpolating the predictive CDF at half-yard bin boundaries.

In [ ]:
example = predictions["q1"].filter(
    (pl.col("game_id") == "2025_13_CIN_BAL")
    & (pl.col("actual_qb_name") == "Joe Burrow")
)
row = (example if example.height else predictions["q1"].head(1)).to_dicts()[0]
line = 200
taus = row["nori_quantile_taus"]
values = row["nori_quantile_values"]
max_yards = max(500, int(np.ceil(max(values) / 5) * 5))
boundaries = np.arange(-0.5, max_yards + 5, 5)
cdf = 1 - np.array([probability_over_line(taus, values, x) for x in boundaries])
mass = np.maximum(0, np.diff(cdf))
centers = boundaries[:-1] + 2.5
p_over = probability_over_line(taus, values, line - 0.5)
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(
    centers,
    100 * mass,
    width=4.5,
    color=np.where(centers >= line, "#33856b", "#c98a66"),
)
ax.axvline(
    line - 0.5, color="#b42318", label=f"{line}+ yards; Nori probability {p_over:.1%}"
)
ax.set(
    xlabel="Final passing yards",
    ylabel="Probability per 5-yard bin (%)",
    title=f"{row['actual_qb_name']} — Week {row['week']} — after Q1",
)
ax.legend()
plt.show()
fig.savefig(OUTPUT_DIR / "passing_yards_distribution.png", bbox_inches="tight")

## 7. Replay the original Kalshi rule
After Q1, select the quoted line whose midpoint is closest to 50¢, then its
higher-edge side. Buy only if probability × $1 − price − fee is at least 10¢.
If Q1 does not buy, select the maximum-edge halftime candidate. Its edge must
reach 10¢ and its probability for the **same line and side** must be at least
the Q1 probability. At most one contract is bought per quarterback-game.

Use the latest minute-close bid/ask between the anchor and decision; exclude
quotes over five minutes old or with spreads over 10¢. YES buys at the YES ask;
NO buys at one minus the YES bid. Modeled taker fees and the extra execution
allowance reduce reported profit. Settlement comes from the public market.
These are **quote-based simulations, not verified fills**.

Missing API responses stop only the market stage. They never become invented
quotes, inferred fills, or a profitable result on an incomplete download.

In [ ]:
if RUN_MARKET_BACKTEST:
    decisions, market_report = run_blog_backtest(
        predictions["q1"],
        predictions["halftime"],
        CACHE_DIR,
        execution_cost=EXECUTION_COST,
    )
else:
    decisions = pl.DataFrame()
    market_report = {"status": "disabled", "roi": None}
print(json.dumps(market_report, indent=2))
(OUTPUT_DIR / "kalshi_status.json").write_text(json.dumps(market_report, indent=2))

In [ ]:
if market_report["status"] == "quote_based_simulation":
    decisions.write_parquet(OUTPUT_DIR / "kalshi_decisions.parquet")
    trades = decisions.filter(pl.col("bet_taken"))
    display(
        trades.select(
            "week",
            "actual_qb_name",
            "selected_horizon",
            "line",
            "side",
            "yes_bid",
            "yes_ask",
            "model_probability",
            "entry_price",
            "fee",
            "expected_net_edge",
            "stressed_capital",
            "stressed_profit",
        ).to_pandas()
    )
    trades.write_csv(OUTPUT_DIR / "kalshi_selected_trades.csv")
else:
    print(
        "Forecasts are saved. No betting return is reported without available market data."
    )

## 8. Keep the provenance with the results
Feature input hashes, pinned model identity, per-week feature selections,
forecast metrics, and the market status are saved beside predictions. The
original strategy was refined on 2025: this is an exploratory reproduction,
not an untouched test or a guarantee of future returns. Public source revisions
and runtime differences can change numbers even with the same algorithm.

In [ ]:
run_manifest = {
    "implementation": "notebook plus feature helper; original algorithm identifier in per-week manifests",
    "package_versions": {p: metadata.version(p) for p in required},
    "weeks": WEEKS,
    "device": DEVICE,
    "execution_cost": EXECUTION_COST,
    "market_status": market_report["status"],
    "resume_verified_local_predictions": RESUME_PREDICTIONS,
    "precomputed_research_predictions": False,
    "input_hash_file": "feature_input_hashes.json",
}
(OUTPUT_DIR / "notebook_run_manifest.json").write_text(
    json.dumps(run_manifest, indent=2)
)
display(
    pd.DataFrame(
        [
            {"file": p.name, "bytes": p.stat().st_size}
            for p in sorted(OUTPUT_DIR.iterdir())
            if p.is_file()
        ]
    )
)
print(f"Outputs: {OUTPUT_DIR.resolve()}")